<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research question

Which pages should be reviewed first for possible content improvement, using signals available before the review decision?

### Decision supported

The analysis supports a practical review-prioritization decision: which pages should be placed near the top of a content-review queue.

The goal is not to claim that a page will definitely improve after intervention. Instead, the goal is to identify pages whose observed search-performance signals make them reasonable candidates for earlier review.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the FlyRank internship warehouse release available for this research task.

The working grain is one page per client per report date, represented by:

- `client_hash_id`
- `content_hash_id`
- `report_date`

The decision cutoff is March 31, 2026. March data is used as the feature window, and April 2026 is used as the outcome window.

The March feature frame contains:

- Google Search Console impressions
- Google Search Console clicks
- Google Search Console CTR
- Google Search Console average position
- GA4 organic sessions

The March feature frame contains 176,738 page-level records.

GA4 sessions are substantially more incomplete than the GSC signals, so this feature is treated cautiously rather than being assumed to be equally reliable for every page.

No client names, URLs, private queries, or other identifying information are included in the public-facing analysis.

Rows or fields that were unavailable for the required feature calculation were excluded from calculations that depended on them. This was done to keep the analysis reproducible rather than silently imputing unavailable business measurements.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print(HF_TOKEN is not None)

True


In [3]:
# Capstone — Step 1: Connect to the FlyRank warehouse

import duckdb
import pandas as pd
import numpy as np

# Create DuckDB connection
con = duckdb.connect()

# Give DuckDB access to the Hugging Face dataset
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# Location of the FlyRank warehouse
FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

print("FlyRank warehouse connection is ready.")

FlyRank warehouse connection is ready.


In [4]:
# Capstone — Step 2: Build the March feature frame

features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions_march,

        SUM(gsc_clicks) AS gsc_clicks_march,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_ctr_march,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position_march,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE NULL
            END
        ) AS ga4_sessions_march

    FROM read_parquet('{FACT}')

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-31'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", features.shape)

display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.001073,6.893301,1.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,NaN
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.535346,3.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.002629,7.435680,2.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.002331,3.871795,2.0


In [5]:
# Capstone — Step 3: Build the March-to-April outcome

outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-01'
                 AND report_date <= DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS march_clicks,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-04-01'
                 AND report_date <= DATE '2026-04-30'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS april_clicks

    FROM read_parquet('{FACT}')

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-04-30'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Outcome frame shape:", outcome.shape)

display(outcome.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Outcome frame shape: (212949, 4)


,client_hash_id,content_hash_id,march_clicks,april_clicks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2.0,2.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,0.0,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,0.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,6.0,30.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,16.0,6.0


In [6]:
# Capstone — Step 4: Join March features with the March-to-April outcome

model_frame = features.merge(
    outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model frame shape:", model_frame.shape)
display(model_frame.head())

Model frame shape: (176738, 9)


,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march,march_clicks,april_clicks
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.001073,6.893301,1.0,7.0,8.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,NaN,0.0,2.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.535346,3.0,6.0,4.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.002629,7.435680,2.0,13.0,8.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.002331,3.871795,2.0,1.0,0.0


In [7]:
# Capstone — Step 5: Create the decline label

model_frame["decline_label"] = (
    model_frame["april_clicks"] < model_frame["march_clicks"]
).astype(int)

print("Decline label counts:")
print(model_frame["decline_label"].value_counts())

print("\nDecline label proportions:")
print(model_frame["decline_label"].value_counts(normalize=True))

Decline label counts:
decline_label
0    131636
1     45102
Name: count, dtype: int64

Decline label proportions:
decline_label
0    0.744809
1    0.255191
Name: proportion, dtype: float64


In [8]:
# Capstone — Step 6: Define features and target

feature_cols = [
    "gsc_impressions_march",
    "gsc_ctr_march",
    "gsc_avg_position_march",
    "ga4_sessions_march"
]

X = model_frame[feature_cols].copy()
y = model_frame["decline_label"].copy()

print("Features used by the model:")
print(feature_cols)

print("\nFeature frame shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing values in model features:")
print(X.isna().sum())

print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Features used by the model:
['gsc_impressions_march', 'gsc_ctr_march', 'gsc_avg_position_march', 'ga4_sessions_march']

Feature frame shape: (176738, 4)
Target shape: (176738,)

Missing values in model features:
gsc_impressions_march          0
gsc_ctr_march                  0
gsc_avg_position_march         0
ga4_sessions_march        112882
dtype: int64

Target distribution:
decline_label
0    0.744809
1    0.255191
Name: proportion, dtype: float64


In [9]:
# Capstone — Step 7: Create a client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

groups = model_frame["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("\nClients appearing in both train and test:")
print(len(set(groups_train) & set(groups_test)))

print("\nTraining decline rate:", y_train.mean())
print("Test decline rate:", y_test.mean())

Training rows: 138310
Test rows: 38428

Training clients: 37
Test clients: 10

Clients appearing in both train and test:
0

Training decline rate: 0.2466416021979611
Test decline rate: 0.2859633600499636


In [10]:
# Capstone — Step 8: Train an interpretable decision tree

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "classifier",
        DecisionTreeClassifier(
            max_depth=3,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [11]:
# Capstone — Step 9: Generate decline probabilities for the held-out test set

test_probabilities = model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(test_probabilities))
print("Minimum probability:", test_probabilities.min())
print("Maximum probability:", test_probabilities.max())

print("\nFirst 10 predicted probabilities:")
print(test_probabilities[:10])

Number of test predictions: 38428
Minimum probability: 0.0
Maximum probability: 0.9402756508422665

First 10 predicted probabilities:
[0.66567757 0.         0.66567757 0.66567757 0.66567757 0.66567757
 0.78794084 0.66567757 0.66567757 0.66567757]


In [12]:
# Capstone — Step 10: Evaluate Precision@20 and Precision@50

test_results = model_frame.iloc[test_idx][
    ["client_hash_id", "content_hash_id", "decline_label"]
].copy()

test_results["decline_probability"] = test_probabilities

# Rank highest-probability pages first
test_results = test_results.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

precision_at_20 = test_results.head(20)["decline_label"].mean()
precision_at_50 = test_results.head(50)["decline_label"].mean()

print("Precision@20:", precision_at_20)
print("Precision@50:", precision_at_50)

print("\nNumber of actual declines in Top 20:",
      test_results.head(20)["decline_label"].sum())

print("Number of actual declines in Top 50:",
      test_results.head(50)["decline_label"].sum())

print("\nTest-set decline base rate:", y_test.mean())

Precision@20: 0.8
Precision@50: 0.86

Number of actual declines in Top 20: 16
Number of actual declines in Top 50: 43

Test-set decline base rate: 0.2859633600499636


In [14]:
# Capstone — Step 11A: Find the saved baseline file

import os

matches = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "baseline_action_score.csv":
            matches.append(os.path.join(root, file))

print("Baseline files found:")
for path in matches:
    print(path)

Baseline files found:


In [15]:
# Capstone — Step 11B: Recreate the rule-based baseline

baseline = model_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march"
    ]
].copy()

# Convert each signal to a percentile.
# Higher percentile = more of the signal.
baseline["impressions_percentile"] = (
    baseline["gsc_impressions_march"].rank(pct=True)
)

baseline["ctr_percentile"] = (
    baseline["gsc_ctr_march"].rank(pct=True)
)

baseline["position_percentile"] = (
    baseline["gsc_avg_position_march"].rank(pct=True)
)

# Higher action score means higher review priority.
# High impressions = more exposure.
# Low CTR = opportunity despite exposure.
# Higher average position value = weaker position.
baseline["baseline_action_score"] = (
    baseline["impressions_percentile"]
    + (1 - baseline["ctr_percentile"])
    + baseline["position_percentile"]
) / 3

print("Baseline created.")
print("Baseline shape:", baseline.shape)

display(
    baseline[
        [
            "client_hash_id",
            "content_hash_id",
            "baseline_action_score"
        ]
    ]
    .sort_values("baseline_action_score", ascending=False)
    .head(10)
)

Baseline created.
Baseline shape: (176738, 9)


,client_hash_id,content_hash_id,baseline_action_score
10194,client_3197e6291363b4db,content_65b8a4998e633d89,0.880680
18082,client_23a62021009f63c4,content_295e883e0e86ca3c,0.880547
17600,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.870718
5009,client_e547b89c05043229,content_4002467a580a7f98,0.870386
103156,client_23a62021009f63c4,content_959d535a9fcc865c,0.867099
17570,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.866395
16664,client_23a62021009f63c4,content_2da022341f8803c3,0.866148
103097,client_23a62021009f63c4,content_e8700175bf54e3d5,0.865779
6454,client_e547b89c05043229,content_6177aad2ded9dee5,0.864616
15690,client_23a62021009f63c4,content_421ad263ab8c8c03,0.864288


In [16]:
# Capstone — Step 12: Compare baseline and learned model

# Get the baseline scores for the same held-out test pages
baseline_test = baseline[
    ["client_hash_id", "content_hash_id", "baseline_action_score"]
].merge(
    model_frame.iloc[test_idx][
        ["client_hash_id", "content_hash_id", "decline_label"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Rank the baseline from highest priority to lowest priority
baseline_test = baseline_test.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

# Calculate baseline Precision@K
baseline_precision_at_20 = (
    baseline_test.head(20)["decline_label"].mean()
)

baseline_precision_at_50 = (
    baseline_test.head(50)["decline_label"].mean()
)

print("Baseline test rows:", len(baseline_test))

print("\nBaseline Precision@20:", baseline_precision_at_20)
print("Baseline Precision@50:", baseline_precision_at_50)

print("\nBaseline actual declines in Top 20:",
      baseline_test.head(20)["decline_label"].sum())

print("Baseline actual declines in Top 50:",
      baseline_test.head(50)["decline_label"].sum())

print("\n--- Learned model ---")

print("Model Precision@20:", precision_at_20)
print("Model Precision@50:", precision_at_50)

print("\nModel actual declines in Top 20:",
      test_results.head(20)["decline_label"].sum())

print("Model actual declines in Top 50:",
      test_results.head(50)["decline_label"].sum())

print("\nTest-set decline base rate:", y_test.mean())

Baseline test rows: 38428

Baseline Precision@20: 0.0
Baseline Precision@50: 0.0

Baseline actual declines in Top 20: 0
Baseline actual declines in Top 50: 0

--- Learned model ---
Model Precision@20: 0.8
Model Precision@50: 0.86

Model actual declines in Top 20: 16
Model actual declines in Top 50: 43

Test-set decline base rate: 0.2859633600499636


In [17]:
# Capstone — Step 13: Audit the baseline Top 20

baseline_top20 = baseline_test.head(20).copy()

display(
    baseline_top20[
        [
            "client_hash_id",
            "content_hash_id",
            "baseline_action_score",
            "decline_label"
        ]
    ]
)

print("\nBaseline Top-20 decline count:",
      baseline_top20["decline_label"].sum())

print("Baseline Top-20 average action score:",
      baseline_top20["baseline_action_score"].mean())

print("Baseline Top-20 decline rate:",
      baseline_top20["decline_label"].mean())

,client_hash_id,content_hash_id,baseline_action_score,decline_label
0,client_73cda7b4e4f265ea,content_3e56218fa52d24b8,0.861097,0
1,client_73cda7b4e4f265ea,content_7e6beaac87f82570,0.853840,0
2,client_73cda7b4e4f265ea,content_8671edba7788e7ea,0.844102,0
3,client_73cda7b4e4f265ea,content_1e5e0d352d07f54c,0.837264,0
4,client_73cda7b4e4f265ea,content_341331526c4b8b4c,0.835120,0
5,client_3f0ce4d44fe94f3d,content_175616d3cc8bbabd,0.827488,0
6,client_3f0ce4d44fe94f3d,content_cc4a274cbff7d06a,0.826893,0
7,client_73cda7b4e4f265ea,content_ab0078baaa3d50b8,0.826089,0
8,client_73cda7b4e4f265ea,content_aff0708a494d78a6,0.825841,0
9,client_73cda7b4e4f265ea,content_8ed57b4607088cc1,0.824900,0



Baseline Top-20 decline count: 0
Baseline Top-20 average action score: 0.8272219801815869
Baseline Top-20 decline rate: 0.0


In [18]:
# Capstone — Step 14: Inspect the signals behind the baseline Top 20

baseline_audit = baseline_top20[
    ["client_hash_id", "content_hash_id", "baseline_action_score"]
].merge(
    model_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions_march",
            "gsc_ctr_march",
            "gsc_avg_position_march",
            "march_clicks",
            "april_clicks",
            "decline_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

display(baseline_audit)

,client_hash_id,content_hash_id,baseline_action_score,gsc_impressions_march,gsc_ctr_march,gsc_avg_position_march,march_clicks,april_clicks,decline_label
0,client_73cda7b4e4f265ea,content_3e56218fa52d24b8,0.861097,9947.0,0.0,47.327435,0.0,0.0,0
1,client_73cda7b4e4f265ea,content_7e6beaac87f82570,0.853840,4643.0,0.0,59.430971,0.0,0.0,0
2,client_73cda7b4e4f265ea,content_8671edba7788e7ea,0.844102,2636.0,0.0,70.834598,0.0,0.0,0
3,client_73cda7b4e4f265ea,content_1e5e0d352d07f54c,0.837264,2310.0,0.0,67.677056,0.0,0.0,0
4,client_73cda7b4e4f265ea,content_341331526c4b8b4c,0.835120,2236.0,0.0,66.374329,0.0,0.0,0
5,client_3f0ce4d44fe94f3d,content_175616d3cc8bbabd,0.827488,1954.0,0.0,62.578301,0.0,1.0,0
6,client_3f0ce4d44fe94f3d,content_cc4a274cbff7d06a,0.826893,3227.0,0.0,42.157112,0.0,0.0,0
7,client_73cda7b4e4f265ea,content_ab0078baaa3d50b8,0.826089,1683.0,0.0,69.137255,0.0,0.0,0
8,client_73cda7b4e4f265ea,content_aff0708a494d78a6,0.825841,2525.0,0.0,48.712475,0.0,0.0,0
9,client_73cda7b4e4f265ea,content_8ed57b4607088cc1,0.824900,1426.0,0.0,79.563114,0.0,0.0,0


In [19]:
# Capstone — Step 15: Check the eligible population

eligible = model_frame["march_clicks"] > 0

print("Total pages:", len(model_frame))
print("Pages with March clicks > 0:", eligible.sum())
print("Pages with March clicks = 0:", (~eligible).sum())

print("\nEligible proportion:", eligible.mean())

print("\nDecline rate among pages with March clicks > 0:")
print(
    model_frame.loc[eligible, "decline_label"].mean()
)

Total pages: 176738
Pages with March clicks > 0: 68837
Pages with March clicks = 0: 107901

Eligible proportion: 0.3894861320146205

Decline rate among pages with March clicks > 0:
0.6551999651350291


In [20]:
# Capstone — Step 16: Create the eligible modeling population

eligible_frame = model_frame[
    model_frame["march_clicks"] > 0
].copy()

print("Eligible modeling population:", len(eligible_frame))

print("\nDecline label counts:")
print(eligible_frame["decline_label"].value_counts())

print("\nDecline label proportions:")
print(eligible_frame["decline_label"].value_counts(normalize=True))

print("\nClients in eligible population:",
      eligible_frame["client_hash_id"].nunique())

Eligible modeling population: 68837

Decline label counts:
decline_label
1    45102
0    23735
Name: count, dtype: int64

Decline label proportions:
decline_label
1    0.6552
0    0.3448
Name: proportion, dtype: float64

Clients in eligible population: 44


In [21]:
# Capstone — Step 17: Define features and target for eligible pages

feature_cols = [
    "gsc_impressions_march",
    "gsc_ctr_march",
    "gsc_avg_position_march",
    "ga4_sessions_march"
]

X = eligible_frame[feature_cols].copy()
y = eligible_frame["decline_label"].copy()

groups = eligible_frame["client_hash_id"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures used:")
print(feature_cols)

print("\nMissing values:")
print(X.isna().sum())

print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Feature matrix shape: (68837, 4)
Target shape: (68837,)

Features used:
['gsc_impressions_march', 'gsc_ctr_march', 'gsc_avg_position_march', 'ga4_sessions_march']

Missing values:
gsc_impressions_march         0
gsc_ctr_march                 0
gsc_avg_position_march        0
ga4_sessions_march        22910
dtype: int64

Target distribution:
decline_label
1    0.6552
0    0.3448
Name: proportion, dtype: float64


In [22]:
# Capstone — Step 18: Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("\nClients in both train and test:",
      len(set(groups_train) & set(groups_test)))

print("\nTraining decline rate:", y_train.mean())
print("Test decline rate:", y_test.mean())

Training rows: 63775
Test rows: 5062

Training clients: 35
Test clients: 9

Clients in both train and test: 0

Training decline rate: 0.6529831438651509
Test decline rate: 0.6831291979454761


In [23]:
# Capstone — Step 19: Train the decision tree model

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [24]:
# Capstone — Step 20: Generate test-set predictions

test_probabilities = model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(test_probabilities))
print("Minimum probability:", test_probabilities.min())
print("Maximum probability:", test_probabilities.max())

print("\nFirst 10 predicted probabilities:")
print(test_probabilities[:10])

Number of test predictions: 5062
Minimum probability: 0.41441883276115743
Maximum probability: 0.9364705882352942

First 10 predicted probabilities:
[0.78000956 0.63501552 0.78000956 0.78000956 0.78000956 0.78000956
 0.63501552 0.78000956 0.63501552 0.63501552]


In [25]:
# Capstone — Step 21: Evaluate model ranking

test_results = eligible_frame.iloc[test_idx][
    [
        "client_hash_id",
        "content_hash_id",
        "decline_label"
    ]
].copy()

test_results["decline_probability"] = test_probabilities

test_results = test_results.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

precision_at_20 = test_results.head(20)["decline_label"].mean()
precision_at_50 = test_results.head(50)["decline_label"].mean()

print("Model Precision@20:", precision_at_20)
print("Model Precision@50:", precision_at_50)

print("\nActual declines in Top 20:",
      test_results.head(20)["decline_label"].sum())

print("Actual declines in Top 50:",
      test_results.head(50)["decline_label"].sum())

print("\nTest-set decline base rate:",
      y_test.mean())

Model Precision@20: 0.8
Model Precision@50: 0.92

Actual declines in Top 20: 16
Actual declines in Top 50: 46

Test-set decline base rate: 0.6831291979454761


In [26]:
# Capstone — Step 22: Build baseline on eligible pages

baseline = eligible_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march"
    ]
].copy()

baseline["impressions_percentile"] = (
    baseline["gsc_impressions_march"].rank(pct=True)
)

baseline["ctr_percentile"] = (
    baseline["gsc_ctr_march"].rank(pct=True)
)

baseline["position_percentile"] = (
    baseline["gsc_avg_position_march"].rank(pct=True)
)

baseline["baseline_action_score"] = (
    baseline["impressions_percentile"]
    + (1 - baseline["ctr_percentile"])
    + baseline["position_percentile"]
) / 3

print("Baseline rows:", len(baseline))

print("\nTop 5 baseline scores:")
print(
    baseline[
        [
            "client_hash_id",
            "content_hash_id",
            "baseline_action_score"
        ]
    ]
    .sort_values("baseline_action_score", ascending=False)
    .head(5)
)

Baseline rows: 68837

Top 5 baseline scores:
                 client_hash_id           content_hash_id  \
103056  client_23a62021009f63c4  content_6aa54d6bbdbf6f24   
103120  client_23a62021009f63c4  content_c367b0ca57f3559b   
17693   client_23a62021009f63c4  content_73aa61dcedebbf30   
15727   client_23a62021009f63c4  content_96e6613b42b52c42   
103668  client_23a62021009f63c4  content_ab91e088440ace78   

        baseline_action_score  
103056               0.995158  
103120               0.995012  
17693                0.994460  
15727                0.994417  
103668               0.993458  


In [27]:
# Capstone — Step 23: Evaluate baseline on the same test pages

baseline_test = baseline[
    [
        "client_hash_id",
        "content_hash_id",
        "baseline_action_score"
    ]
].merge(
    eligible_frame.iloc[test_idx][
        [
            "client_hash_id",
            "content_hash_id",
            "decline_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

baseline_test = baseline_test.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

baseline_precision_at_20 = (
    baseline_test.head(20)["decline_label"].mean()
)

baseline_precision_at_50 = (
    baseline_test.head(50)["decline_label"].mean()
)

print("Baseline test rows:", len(baseline_test))

print("\nBaseline Precision@20:",
      baseline_precision_at_20)

print("Baseline Precision@50:",
      baseline_precision_at_50)

print("\nActual declines in baseline Top 20:",
      baseline_test.head(20)["decline_label"].sum())

print("Actual declines in baseline Top 50:",
      baseline_test.head(50)["decline_label"].sum())

print("\nTest-set decline base rate:",
      y_test.mean())

Baseline test rows: 5062

Baseline Precision@20: 0.7
Baseline Precision@50: 0.64

Actual declines in baseline Top 20: 14
Actual declines in baseline Top 50: 32

Test-set decline base rate: 0.6831291979454761


In [28]:
# Capstone — Step 24: Inspect the model's Top 20 review candidates

top20_model = test_results.head(20).copy()

top20_model = top20_model.merge(
    eligible_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions_march",
            "gsc_ctr_march",
            "gsc_avg_position_march",
            "ga4_sessions_march",
            "march_clicks",
            "april_clicks"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(
    top20_model[
        [
            "client_hash_id",
            "content_hash_id",
            "decline_probability",
            "decline_label",
            "gsc_impressions_march",
            "gsc_ctr_march",
            "gsc_avg_position_march",
            "ga4_sessions_march",
            "march_clicks",
            "april_clicks"
        ]
    ].to_string(index=False)
)

         client_hash_id          content_hash_id  decline_probability  decline_label  gsc_impressions_march  gsc_ctr_march  gsc_avg_position_march  ga4_sessions_march  march_clicks  april_clicks
client_f623b01661d4bfe4 content_b029e9f500af2a42             0.936471              1                    4.0       0.250000               10.500000                 NaN           1.0           0.0
client_f623b01661d4bfe4 content_6bd6b23eb02abe1b             0.936471              1                    1.0       1.000000                0.000000                 0.0           1.0           0.0
client_f623b01661d4bfe4 content_4693aa2f65099f3c             0.936471              0                    9.0       0.222222               40.333333                 2.0           2.0           2.0
client_f623b01661d4bfe4 content_23c2164b794a058c             0.936471              1                    7.0       0.142857                4.571429                 2.0           1.0           0.0
client_f623b01661d4bfe4 c

In [29]:
# Capstone — Step 25: Inspect the learned decision tree

from sklearn.tree import export_text

tree = model.named_steps["classifier"]

tree_rules = export_text(
    tree,
    feature_names=feature_cols
)

print(tree_rules)

|--- gsc_ctr_march <= 0.00
|   |--- gsc_ctr_march <= 0.00
|   |   |--- gsc_impressions_march <= 7582.00
|   |   |   |--- class: 0
|   |   |--- gsc_impressions_march >  7582.00
|   |   |   |--- class: 1
|   |--- gsc_ctr_march >  0.00
|   |   |--- gsc_impressions_march <= 4430.50
|   |   |   |--- class: 1
|   |   |--- gsc_impressions_march >  4430.50
|   |   |   |--- class: 1
|--- gsc_ctr_march >  0.00
|   |--- gsc_impressions_march <= 136.50
|   |   |--- gsc_ctr_march <= 0.07
|   |   |   |--- class: 1
|   |   |--- gsc_ctr_march >  0.07
|   |   |   |--- class: 1
|   |--- gsc_impressions_march >  136.50
|   |   |--- gsc_ctr_march <= 0.00
|   |   |   |--- class: 1
|   |   |--- gsc_ctr_march >  0.00
|   |   |   |--- class: 1



In [30]:
# Capstone — Step 26: Inspect probability groups

probability_summary = (
    test_results
    .groupby("decline_probability")
    .agg(
        pages=("decline_label", "size"),
        actual_declines=("decline_label", "sum"),
        decline_rate=("decline_label", "mean")
    )
    .sort_values("decline_probability", ascending=False)
)

print(probability_summary.to_string())

                     pages  actual_declines  decline_rate
decline_probability                                      
0.936471               218              201      0.922018
0.780010               903              719      0.796235
0.678214              1605             1029      0.641121
0.662260                14               10      0.714286
0.635016              2061             1364      0.661815
0.528315               215              114      0.530233
0.516941                 5                3      0.600000
0.414419                41               18      0.439024


In [31]:
# Capstone — Step 27: Compare ranking precision at different queue sizes

queue_sizes = [20, 50, 100, 200]

print("Queue size | Baseline Precision | Model Precision")
print("-" * 52)

for k in queue_sizes:
    baseline_precision = baseline_test.head(k)["decline_label"].mean()
    model_precision = test_results.head(k)["decline_label"].mean()

    print(
        f"{k:10d} | "
        f"{baseline_precision:18.3f} | "
        f"{model_precision:16.3f}"
    )

Queue size | Baseline Precision | Model Precision
----------------------------------------------------
        20 |              0.700 |            0.800
        50 |              0.640 |            0.920
       100 |              0.660 |            0.920
       200 |              0.670 |            0.915


In [32]:
# Capstone — Step 28: Check robustness across different client-grouped splits

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

split_seeds = [42, 7, 21, 99, 123]

robustness_results = []

for seed in split_seeds:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=seed
    )

    train_idx_r, test_idx_r = next(
        splitter.split(X, y, groups=groups)
    )

    X_train_r = X.iloc[train_idx_r]
    X_test_r = X.iloc[test_idx_r]

    y_train_r = y.iloc[train_idx_r]
    y_test_r = y.iloc[test_idx_r]

    model_r = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(
            max_depth=3,
            random_state=42
        ))
    ])

    model_r.fit(X_train_r, y_train_r)

    probabilities_r = model_r.predict_proba(X_test_r)[:, 1]

    results_r = eligible_frame.iloc[test_idx_r][
        ["client_hash_id", "content_hash_id", "decline_label"]
    ].copy()

    results_r["decline_probability"] = probabilities_r

    results_r = results_r.sort_values(
        "decline_probability",
        ascending=False
    )

    p20 = results_r.head(20)["decline_label"].mean()
    p50 = results_r.head(50)["decline_label"].mean()
    p100 = results_r.head(100)["decline_label"].mean()

    robustness_results.append({
        "seed": seed,
        "test_pages": len(test_idx_r),
        "test_decline_rate": y_test_r.mean(),
        "precision_at_20": p20,
        "precision_at_50": p50,
        "precision_at_100": p100
    })

robustness_df = pd.DataFrame(robustness_results)

print(robustness_df.to_string(index=False))

print("\nAverage precision:")
print(
    robustness_df[
        ["precision_at_20", "precision_at_50", "precision_at_100"]
    ].mean()
)

print("\nStandard deviation:")
print(
    robustness_df[
        ["precision_at_20", "precision_at_50", "precision_at_100"]
    ].std()
)

 seed  test_pages  test_decline_rate  precision_at_20  precision_at_50  precision_at_100
   42        5062           0.683129             0.80             0.92              0.92
    7       20513           0.598596             0.95             0.94              0.93
   21        6317           0.692576             0.90             0.90              0.88
   99        2299           0.838191             0.85             0.90              0.89
  123       22865           0.691712             0.95             0.98              0.96

Average precision:
precision_at_20     0.890
precision_at_50     0.928
precision_at_100    0.916
dtype: float64

Standard deviation:
precision_at_20     0.065192
precision_at_50     0.033466
precision_at_100    0.032094
dtype: float64


In [33]:
# Capstone — Step 29: Inspect model feature importance

tree = model.named_steps["classifier"]

feature_importance = pd.Series(
    tree.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("Feature importance:")
print(feature_importance)

Feature importance:
gsc_ctr_march             0.630795
gsc_impressions_march     0.369205
gsc_avg_position_march    0.000000
ga4_sessions_march        0.000000
dtype: float64


In [34]:
# Capstone — Step 30: Calculate Average Precision

from sklearn.metrics import average_precision_score

model_average_precision = average_precision_score(
    y_test,
    test_probabilities
)

baseline_average_precision = average_precision_score(
    baseline_test["decline_label"],
    baseline_test["baseline_action_score"]
)

print(f"Model Average Precision: {model_average_precision:.4f}")
print(f"Baseline Average Precision: {baseline_average_precision:.4f}")
print(f"Test-set decline base rate: {y_test.mean():.4f}")

Model Average Precision: 0.7388
Baseline Average Precision: 0.6808
Test-set decline base rate: 0.6831


In [35]:
# Capstone — Step 31: Inspect decline-rate variation across clients

client_decline_rates = (
    eligible_frame
    .groupby("client_hash_id")
    .agg(
        pages=("decline_label", "size"),
        declines=("decline_label", "sum"),
        decline_rate=("decline_label", "mean")
    )
    .sort_values("decline_rate", ascending=False)
)

print("Number of clients:", len(client_decline_rates))

print("\nClient decline-rate summary:")
print(
    client_decline_rates["decline_rate"].describe()
)

print("\nHighest client decline rates:")
print(
    client_decline_rates.head(10).to_string()
)

print("\nLowest client decline rates:")
print(
    client_decline_rates.tail(10).to_string()
)

Number of clients: 44

Client decline-rate summary:
count    44.000000
mean      0.744650
std       0.199347
min       0.078740
25%       0.597751
50%       0.773403
75%       0.909353
max       1.000000
Name: decline_rate, dtype: float64

Highest client decline rates:
                         pages  declines  decline_rate
client_hash_id                                        
client_810019792c9b8efc      6         6      1.000000
client_8ae2bfb5aa1ffa1e      4         4      1.000000
client_8dbf3abdf07569e0      2         2      1.000000
client_59256b0571e0c970      1         1      1.000000
client_ba65e80a1116ae41      1         1      1.000000
client_e00b29e582949543      2         2      1.000000
client_2b4306c3ed003f01     25        24      0.960000
client_795153d5b7850ccf     23        22      0.956522
client_ff644d8251367cbb    818       772      0.943765
client_9d54435aabd95a6c     12        11      0.916667

Lowest client decline rates:
                         pages  declines

In [36]:
# Capstone — Step 32: Compare model and baseline across client splits

split_comparison = []

for seed in split_seeds:

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=seed
    )

    train_idx_r, test_idx_r = next(
        splitter.split(X, y, groups=groups)
    )

    X_train_r = X.iloc[train_idx_r]
    X_test_r = X.iloc[test_idx_r]

    y_train_r = y.iloc[train_idx_r]
    y_test_r = y.iloc[test_idx_r]

    model_r = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(
            max_depth=3,
            random_state=42
        ))
    ])

    model_r.fit(X_train_r, y_train_r)

    model_prob_r = model_r.predict_proba(X_test_r)[:, 1]

    model_ranked_r = eligible_frame.iloc[test_idx_r][
        [
            "client_hash_id",
            "content_hash_id",
            "decline_label"
        ]
    ].copy()

    model_ranked_r["model_probability"] = model_prob_r

    model_ranked_r = model_ranked_r.sort_values(
        "model_probability",
        ascending=False
    )

    model_p50 = model_ranked_r.head(50)["decline_label"].mean()

    baseline_ranked_r = baseline[
        [
            "client_hash_id",
            "content_hash_id",
            "baseline_action_score"
        ]
    ].merge(
        eligible_frame.iloc[test_idx_r][
            [
                "client_hash_id",
                "content_hash_id",
                "decline_label"
            ]
        ],
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )

    baseline_ranked_r = baseline_ranked_r.sort_values(
        "baseline_action_score",
        ascending=False
    )

    baseline_p50 = baseline_ranked_r.head(50)["decline_label"].mean()

    split_comparison.append({
        "seed": seed,
        "model_precision_at_50": model_p50,
        "baseline_precision_at_50": baseline_p50,
        "model_minus_baseline": model_p50 - baseline_p50
    })

split_comparison_df = pd.DataFrame(split_comparison)

print(split_comparison_df.to_string(index=False))

print("\nAverage model Precision@50:",
      split_comparison_df["model_precision_at_50"].mean())

print("Average baseline Precision@50:",
      split_comparison_df["baseline_precision_at_50"].mean())

print("Average improvement:",
      split_comparison_df["model_minus_baseline"].mean())

print(
    "\nSplits where model beats baseline:",
    (
        split_comparison_df["model_minus_baseline"] > 0
    ).sum(),
    "out of",
    len(split_comparison_df)
)

 seed  model_precision_at_50  baseline_precision_at_50  model_minus_baseline
   42                   0.92                      0.64                  0.28
    7                   0.94                      0.62                  0.32
   21                   0.90                      0.50                  0.40
   99                   0.90                      0.80                  0.10
  123                   0.98                      0.76                  0.22

Average model Precision@50: 0.9279999999999999
Average baseline Precision@50: 0.664
Average improvement: 0.264

Splits where model beats baseline: 5 out of 5


In [37]:
# Capstone — Step 33: Inspect model ranking errors

error_analysis = test_results.copy()

error_analysis["prediction"] = (
    error_analysis["decline_probability"] >= 0.50
).astype(int)

false_positives = error_analysis[
    (error_analysis["prediction"] == 1) &
    (error_analysis["decline_label"] == 0)
].copy()

false_negatives = error_analysis[
    (error_analysis["prediction"] == 0) &
    (error_analysis["decline_label"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nHighest-ranked false positives:")
print(
    false_positives.head(10).to_string(index=False)
)

print("\nLowest-ranked false negatives:")
print(
    false_negatives.sort_values(
        "decline_probability",
        ascending=True
    ).head(10).to_string(index=False)
)

False positives: 1581
False negatives: 18

Highest-ranked false positives:
         client_hash_id          content_hash_id  decline_label  decline_probability  prediction
client_f623b01661d4bfe4 content_4693aa2f65099f3c              0             0.936471           1
client_f623b01661d4bfe4 content_910bad0d28839701              0             0.936471           1
client_f623b01661d4bfe4 content_8d434a68a6a804aa              0             0.936471           1
client_2094c6eb080311d5 content_f2d1fe79601a06f5              0             0.936471           1
client_f623b01661d4bfe4 content_3e3b42aaddb358c6              0             0.936471           1
client_f623b01661d4bfe4 content_5eea911094386204              0             0.936471           1
client_cd12bcfd98942aa1 content_a27a4cdc0e9ec4ab              0             0.936471           1
client_cd12bcfd98942aa1 content_b4273706aceafa40              0             0.936471           1
client_2094c6eb080311d5 content_6414a9dda4cac1ff    

In [38]:
# Capstone — Step 34: Inspect errors inside the Top-50 review queue

top50 = test_results.head(50).copy()

top50["result"] = np.where(
    top50["decline_label"] == 1,
    "Correct priority",
    "False positive"
)

print("Top-50 queue:")
print(
    top50[
        [
            "client_hash_id",
            "content_hash_id",
            "decline_label",
            "decline_probability",
            "result"
        ]
    ].to_string(index=False)
)

print("\nTop-50 result counts:")
print(top50["result"].value_counts())

print("\nTop-50 result proportions:")
print(top50["result"].value_counts(normalize=True))

Top-50 queue:
         client_hash_id          content_hash_id  decline_label  decline_probability           result
client_f623b01661d4bfe4 content_b029e9f500af2a42              1             0.936471 Correct priority
client_f623b01661d4bfe4 content_6bd6b23eb02abe1b              1             0.936471 Correct priority
client_f623b01661d4bfe4 content_4693aa2f65099f3c              0             0.936471   False positive
client_f623b01661d4bfe4 content_23c2164b794a058c              1             0.936471 Correct priority
client_f623b01661d4bfe4 content_c23f2f8767e964ff              1             0.936471 Correct priority
client_f623b01661d4bfe4 content_a03376db17dd976a              1             0.936471 Correct priority
client_f623b01661d4bfe4 content_9c9cde1341e0af5d              1             0.936471 Correct priority
client_f623b01661d4bfe4 content_d37466eb5c81306f              1             0.936471 Correct priority
client_cd12bcfd98942aa1 content_d2ab87c8973c8fd1              1     

In [39]:
# Capstone — Step 35: Create model priority tiers

final_queue = test_results.copy()

final_queue["priority_tier"] = pd.cut(
    final_queue["decline_probability"],
    bins=[-np.inf, 0.60, 0.70, 0.80, np.inf],
    labels=[
        "Lower priority",
        "Medium priority",
        "High priority",
        "Highest priority"
    ]
)

print("Priority tier counts:")
print(final_queue["priority_tier"].value_counts().sort_index())

print("\nPriority tier decline rates:")
print(
    final_queue.groupby(
        "priority_tier",
        observed=False
    )["decline_label"].mean()
)

print("\nTop-50 priority tiers:")
print(
    final_queue.head(50)["priority_tier"].value_counts()
)

Priority tier counts:
priority_tier
Lower priority       261
Medium priority     3680
High priority        903
Highest priority     218
Name: count, dtype: int64

Priority tier decline rates:
priority_tier
Lower priority      0.517241
Medium priority     0.652989
High priority       0.796235
Highest priority    0.922018
Name: decline_label, dtype: float64

Top-50 priority tiers:
priority_tier
Highest priority    50
Lower priority       0
Medium priority      0
High priority        0
Name: count, dtype: int64


In [40]:
# Capstone — Step 36: Final evaluation summary

final_summary = {
    "eligible_pages": len(eligible_frame),
    "eligible_decline_rate": y.mean(),

    "test_pages_seed_42": len(y_test),
    "test_decline_rate_seed_42": y_test.mean(),

    "model_precision_at_20": precision_at_20,
    "model_precision_at_50": precision_at_50,

    "baseline_precision_at_20": baseline_precision_at_20,
    "baseline_precision_at_50": baseline_precision_at_50,

    "model_average_precision": 0.7388,
    "baseline_average_precision": 0.6808,

    "five_split_model_precision_at_50": 0.928,
    "five_split_baseline_precision_at_50": 0.664,
    "five_split_average_improvement": 0.264,

    "top50_correct_priorities": 46,
    "top50_false_positives": 4,

    "highest_tier_decline_rate": 0.922018,
    "high_tier_decline_rate": 0.796235,
    "medium_tier_decline_rate": 0.652989,
    "lower_tier_decline_rate": 0.517241
}

for key, value in final_summary.items():
    print(f"{key}: {value}")

eligible_pages: 68837
eligible_decline_rate: 0.6551999651350291
test_pages_seed_42: 5062
test_decline_rate_seed_42: 0.6831291979454761
model_precision_at_20: 0.8
model_precision_at_50: 0.92
baseline_precision_at_20: 0.7
baseline_precision_at_50: 0.64
model_average_precision: 0.7388
baseline_average_precision: 0.6808
five_split_model_precision_at_50: 0.928
five_split_baseline_precision_at_50: 0.664
five_split_average_improvement: 0.264
top50_correct_priorities: 46
top50_false_positives: 4
highest_tier_decline_rate: 0.922018
high_tier_decline_rate: 0.796235
medium_tier_decline_rate: 0.652989
lower_tier_decline_rate: 0.517241


In [41]:
# Capstone — Step 37: Recalculate Average Precision

from sklearn.metrics import average_precision_score

model_ap = average_precision_score(
    y_test,
    test_probabilities
)

baseline_ap = average_precision_score(
    baseline_test["decline_label"],
    baseline_test["baseline_action_score"]
)

print(f"Model Average Precision: {model_ap:.4f}")
print(f"Baseline Average Precision: {baseline_ap:.4f}")
print(f"Test-set decline base rate: {y_test.mean():.4f}")
print(f"Model improvement over baseline: {model_ap - baseline_ap:.4f}")

Model Average Precision: 0.7388
Baseline Average Precision: 0.6808
Test-set decline base rate: 0.6831
Model improvement over baseline: 0.0580


In [43]:
# Capstone — Final artifact: ranked content-review queue

final_queue = test_results.copy()

# Add the March features needed for editorial interpretation
final_queue = final_queue.merge(
    eligible_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions_march",
            "gsc_ctr_march",
            "gsc_avg_position_march",
            "ga4_sessions_march"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Create priority tiers from the model score
def assign_priority_tier(probability):
    if probability >= 0.90:
        return "Highest priority"
    elif probability >= 0.75:
        return "High priority"
    elif probability >= 0.60:
        return "Medium priority"
    else:
        return "Lower priority"

final_queue["priority_tier"] = (
    final_queue["decline_probability"]
    .apply(assign_priority_tier)
)

# Add a simple explanation for why the page was prioritized
def make_reason(row):
    reasons = []

    if row["gsc_ctr_march"] <= 0:
        reasons.append("zero CTR")
    elif row["gsc_ctr_march"] < 0.05:
        reasons.append("low CTR")

    if row["gsc_impressions_march"] >= 1000:
        reasons.append("high impressions")

    if row["gsc_avg_position_march"] > 10:
        reasons.append("weak average position")

    if not reasons:
        reasons.append("model-ranked signal combination")

    return "; ".join(reasons)

final_queue["reason_code"] = final_queue.apply(
    make_reason,
    axis=1
)

# Rank the queue
final_queue = final_queue.sort_values(
    ["decline_probability", "gsc_impressions_march"],
    ascending=[False, False]
).reset_index(drop=True)

final_queue["rank"] = np.arange(1, len(final_queue) + 1)

# Select the final public-facing artifact columns
final_queue = final_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "decline_probability",
        "priority_tier",
        "reason_code",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
        "ga4_sessions_march"
    ]
]

# Save artifact
import os

OUTPUT_DIR = "/content/ML_Internship/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "capstone_ranked_review_queue.csv"
)

final_queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Final ranked queue created.")
print("Rows:", len(final_queue))
print("Columns:", len(final_queue.columns))
print()
print("Priority tier counts:")
print(final_queue["priority_tier"].value_counts())
print()
print("Top 10 pages:")
display(final_queue.head(10))
print()
print("Saved to:")
print(OUTPUT_PATH)

Final ranked queue created.
Rows: 5062
Columns: 10

Priority tier counts:
priority_tier
Medium priority     3680
High priority        903
Lower priority       261
Highest priority     218
Name: count, dtype: int64

Top 10 pages:


,rank,client_hash_id,content_hash_id,decline_probability,priority_tier,reason_code,gsc_impressions_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march
0,1,client_2094c6eb080311d5,content_6546112a234cb98d,0.936471,Highest priority,model-ranked signal combination,78.0,0.076923,5.141026,7.0
1,2,client_f623b01661d4bfe4,content_4663d6efb6744b2d,0.936471,Highest priority,model-ranked signal combination,73.0,0.082192,7.438356,5.0
2,3,client_f623b01661d4bfe4,content_d141093a88e6133f,0.936471,Highest priority,model-ranked signal combination,66.0,0.106061,4.090909,6.0
3,4,client_9958f0a7ae1df715,content_26b315a13df7951f,0.936471,Highest priority,weak average position,63.0,0.111111,14.269841,6.0
4,5,client_3f0ce4d44fe94f3d,content_bacfb410eac212da,0.936471,Highest priority,model-ranked signal combination,54.0,0.074074,6.444444,1.0
5,6,client_f623b01661d4bfe4,content_00914d6da1fccc09,0.936471,Highest priority,weak average position,46.0,0.086957,42.934783,6.0
6,7,client_f623b01661d4bfe4,content_ab343455aa5aee3f,0.936471,Highest priority,model-ranked signal combination,42.0,0.095238,8.023810,2.0
7,8,client_9958f0a7ae1df715,content_5d531fb55b4df090,0.936471,Highest priority,weak average position,39.0,0.076923,16.538462,7.0
8,9,client_f623b01661d4bfe4,content_3e3b42aaddb358c6,0.936471,Highest priority,model-ranked signal combination,38.0,0.078947,7.500000,5.0
9,10,client_f623b01661d4bfe4,content_b0040fecd9a1b4d8,0.936471,Highest priority,weak average position,35.0,0.114286,27.942857,4.0



Saved to:
/content/ML_Internship/work/outputs/capstone_ranked_review_queue.csv


In [44]:
# Capstone — Final full-population review queue

# Train the selected model on all eligible historical pages
final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

final_model.fit(
    eligible_frame[feature_cols],
    eligible_frame["decline_label"]
)

# Generate model scores for all eligible pages
all_probabilities = final_model.predict_proba(
    eligible_frame[feature_cols]
)[:, 1]

# Start the final queue with the page identifiers and March features
final_queue_all = eligible_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
        "ga4_sessions_march"
    ]
].copy()

final_queue_all["decline_probability"] = all_probabilities


# Assign priority tiers
def assign_priority_tier(probability):
    if probability >= 0.90:
        return "Highest priority"
    elif probability >= 0.75:
        return "High priority"
    elif probability >= 0.60:
        return "Medium priority"
    else:
        return "Lower priority"


final_queue_all["priority_tier"] = (
    final_queue_all["decline_probability"]
    .apply(assign_priority_tier)
)


# Add simple contextual reason codes
def make_reason(row):
    reasons = []

    if row["gsc_ctr_march"] <= 0:
        reasons.append("zero CTR")
    elif row["gsc_ctr_march"] < 0.05:
        reasons.append("low CTR")

    if row["gsc_impressions_march"] >= 1000:
        reasons.append("high impressions")

    if row["gsc_avg_position_march"] > 10:
        reasons.append("weak average position")

    if not reasons:
        reasons.append("model-ranked signal combination")

    return "; ".join(reasons)


final_queue_all["reason_code"] = final_queue_all.apply(
    make_reason,
    axis=1
)


# Sort into the final review order
final_queue_all = final_queue_all.sort_values(
    ["decline_probability", "gsc_impressions_march"],
    ascending=[False, False]
).reset_index(drop=True)

final_queue_all["rank"] = np.arange(
    1,
    len(final_queue_all) + 1
)


# Keep only the public-facing artifact columns
final_queue_all = final_queue_all[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "decline_probability",
        "priority_tier",
        "reason_code",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
        "ga4_sessions_march"
    ]
]


# Save the final full-population queue
OUTPUT_PATH_ALL = (
    "/content/ML_Internship/work/outputs/"
    "capstone_final_ranked_review_queue.csv"
)

final_queue_all.to_csv(
    OUTPUT_PATH_ALL,
    index=False
)


print("Final full-population queue created.")
print("Rows:", len(final_queue_all))
print("Columns:", len(final_queue_all.columns))
print()

print("Priority tier counts:")
print(final_queue_all["priority_tier"].value_counts())
print()

print("Top 10 pages:")
display(final_queue_all.head(10))
print()

print("Saved to:")
print(OUTPUT_PATH_ALL)

Final full-population queue created.
Rows: 68837
Columns: 10

Priority tier counts:
priority_tier
Medium priority     53168
Lower priority       9125
High priority        5051
Highest priority     1493
Name: count, dtype: int64

Top 10 pages:


,rank,client_hash_id,content_hash_id,decline_probability,priority_tier,reason_code,gsc_impressions_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march
0,1,client_23a62021009f63c4,content_7d87c69eaba32c8c,0.93436,Highest priority,model-ranked signal combination,126.0,0.103175,4.698413,15.0
1,2,client_3ffa76342f366962,content_78de668c0f317f11,0.93436,Highest priority,model-ranked signal combination,121.0,0.074380,3.338843,7.0
2,3,client_08a6a72ff48e62c0,content_f78eb098f6dbf006,0.93436,Highest priority,model-ranked signal combination,112.0,0.080357,3.526786,NaN
3,4,client_3ffa76342f366962,content_f27964d7fd33509f,0.93436,Highest priority,model-ranked signal combination,102.0,0.078431,3.019608,7.0
4,5,client_3ffa76342f366962,content_2edf5c5651a9646a,0.93436,Highest priority,model-ranked signal combination,91.0,0.076923,3.835165,8.0
5,6,client_3ffa76342f366962,content_9b46fd2fd4549f56,0.93436,Highest priority,model-ranked signal combination,89.0,0.101124,2.764045,8.0
6,7,client_3ffa76342f366962,content_caf756e63ca7f979,0.93436,Highest priority,model-ranked signal combination,82.0,0.097561,5.829268,5.0
7,8,client_3ffa76342f366962,content_924fcfa2714a8454,0.93436,Highest priority,model-ranked signal combination,80.0,0.162500,2.800000,10.0
8,9,client_2094c6eb080311d5,content_6546112a234cb98d,0.93436,Highest priority,model-ranked signal combination,78.0,0.076923,5.141026,7.0
9,10,client_f623b01661d4bfe4,content_4663d6efb6744b2d,0.93436,Highest priority,model-ranked signal combination,73.0,0.082192,7.438356,5.0



Saved to:
/content/ML_Internship/work/outputs/capstone_final_ranked_review_queue.csv


In [45]:
# Capstone — Model-grounded explanation labels

# Inspect which features the final tree actually uses
final_classifier = final_model.named_steps["classifier"]

print("Final model feature importances:")
for feature, importance in zip(feature_cols, final_classifier.feature_importances_):
    print(f"{feature}: {importance:.4f}")


# Create explanation labels based on the signals the model actually learned
def make_model_reason(row):
    reasons = []

    # CTR is the strongest learned signal
    if row["gsc_ctr_march"] < 0.05:
        reasons.append("low CTR signal")

    # Impressions are the second learned signal
    if row["gsc_impressions_march"] > 100:
        reasons.append("higher exposure signal")

    # Position and GA4 were not used by the final tree,
    # so we do not describe them as model drivers.

    if not reasons:
        reasons.append("learned CTR and exposure pattern")

    return "; ".join(reasons)


final_queue_all["reason_code"] = final_queue_all.apply(
    make_model_reason,
    axis=1
)


# Re-save the improved final artifact
OUTPUT_PATH_ALL = (
    "/content/ML_Internship/work/outputs/"
    "capstone_final_ranked_review_queue.csv"
)

final_queue_all.to_csv(
    OUTPUT_PATH_ALL,
    index=False
)


print()
print("Improved final queue saved.")
print("Rows:", len(final_queue_all))
print("Columns:", len(final_queue_all.columns))
print()
print("Reason code counts:")
print(final_queue_all["reason_code"].value_counts())
print()
print("Top 10 pages:")
display(final_queue_all.head(10))
print()
print("Saved to:")
print(OUTPUT_PATH_ALL)

Final model feature importances:
gsc_impressions_march: 0.3962
gsc_ctr_march: 0.6038
gsc_avg_position_march: 0.0000
ga4_sessions_march: 0.0000

Improved final queue saved.
Rows: 68837
Columns: 10

Reason code counts:
reason_code
low CTR signal; higher exposure signal    63578
low CTR signal                             3241
learned CTR and exposure pattern           1970
higher exposure signal                       48
Name: count, dtype: int64

Top 10 pages:


,rank,client_hash_id,content_hash_id,decline_probability,priority_tier,reason_code,gsc_impressions_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march
0,1,client_23a62021009f63c4,content_7d87c69eaba32c8c,0.93436,Highest priority,higher exposure signal,126.0,0.103175,4.698413,15.0
1,2,client_3ffa76342f366962,content_78de668c0f317f11,0.93436,Highest priority,higher exposure signal,121.0,0.074380,3.338843,7.0
2,3,client_08a6a72ff48e62c0,content_f78eb098f6dbf006,0.93436,Highest priority,higher exposure signal,112.0,0.080357,3.526786,NaN
3,4,client_3ffa76342f366962,content_f27964d7fd33509f,0.93436,Highest priority,higher exposure signal,102.0,0.078431,3.019608,7.0
4,5,client_3ffa76342f366962,content_2edf5c5651a9646a,0.93436,Highest priority,learned CTR and exposure pattern,91.0,0.076923,3.835165,8.0
5,6,client_3ffa76342f366962,content_9b46fd2fd4549f56,0.93436,Highest priority,learned CTR and exposure pattern,89.0,0.101124,2.764045,8.0
6,7,client_3ffa76342f366962,content_caf756e63ca7f979,0.93436,Highest priority,learned CTR and exposure pattern,82.0,0.097561,5.829268,5.0
7,8,client_3ffa76342f366962,content_924fcfa2714a8454,0.93436,Highest priority,learned CTR and exposure pattern,80.0,0.162500,2.800000,10.0
8,9,client_2094c6eb080311d5,content_6546112a234cb98d,0.93436,Highest priority,learned CTR and exposure pattern,78.0,0.076923,5.141026,7.0
9,10,client_f623b01661d4bfe4,content_4663d6efb6744b2d,0.93436,Highest priority,learned CTR and exposure pattern,73.0,0.082192,7.438356,5.0



Saved to:
/content/ML_Internship/work/outputs/capstone_final_ranked_review_queue.csv


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Assumptions

The analysis assumes that March search-performance signals are available before the review decision and can therefore be used as input features.

The resulting score is treated as decision support, not as a guarantee that a page will decline or improve.

### Features

The main March features are:

- `gsc_impressions_march`
- `gsc_ctr_march`
- `gsc_avg_position_march`
- `sessions_organic_march`

These features were selected because the Week 4 signal audit found directional relationships between these signals and the April outcome.

### Label definition

The outcome is defined using the change in clicks from the March feature window to the April outcome window.

The label identifies whether a page experienced the defined decline condition during the outcome window.

The label is only used for model evaluation/training and is not available when generating the March-based review queue.

### Baseline

The baseline is a simple rule-based action score derived from the observed March search signals.

The baseline is intentionally simple so that the learned model can be compared against a transparent reference rather than against an arbitrary or already-optimized system.

### Validation design

The model and baseline are evaluated on the same held-out data and using the same evaluation metric.

The primary review-queue metric is Precision@K, where K represents the number of highest-priority pages reviewed.

This measures how many of the pages selected near the top of the queue match the defined outcome.

### Leakage checks

A leakage check was performed before model evaluation.

The feature columns use March data only, while the outcome is measured using April data.

The leakage check passed because no future/outcome columns were used as model features.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The learned model was evaluated against the rule-based baseline on the same held-out split and using the same Precision@K metric.

| Approach | Precision@20 | Precision@50 |
|---|---:|---:|
| Baseline | [actual value] | [actual value] |
| Learned model | [actual value] | [actual value] |

The comparison shows whether the learned model provides additional value over the transparent baseline for the intended review-queue decision.

## 5. Limitations

*What this work cannot claim.*

This work has several important limitations.

1. The analysis is based on observational historical data. The relationships found between signals and outcomes should therefore be interpreted as directional rather than causal.

2. A high-ranked page is a review candidate, not proof that changing the page will improve performance.

3. The outcome is based on changes in clicks between the March and April windows. This does not directly measure whether a content intervention was successful.

4. GA4 session data has substantial missingness in the feature frame, so conclusions involving this signal should be treated cautiously.

5. Search performance can be affected by factors outside the page itself, including seasonality, search demand, competition, algorithm changes, and measurement differences.

6. Precision@K measures the usefulness of the ranked review queue but does not measure the business impact of actually reviewing or improving a page.

7. The results should not be interpreted as evidence that the model will perform identically on future releases or different clients without further validation.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The final output is a ranked review queue rather than a claim that every selected page requires an immediate content change.

### Recommended action playbook

| Priority | Recommended action |
|---|---|
| High | Review the page first for content quality, search intent alignment, and recent performance changes. |
| Medium | Review after the highest-priority pages, especially when multiple search signals indicate weaker performance. |
| Lower | Defer review unless additional business or content context suggests otherwise. |

The ranked queue should be used as a decision-support tool. Analysts or content teams should inspect the underlying signals and page context before making an intervention.

Reason codes are included with the ranked output so that a recommendation is explainable rather than being based only on an opaque score.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed research page will include the following artifacts:

1. **Signal audit chart**
   - Shows the relationship between key March signals and the April outcome.
   - Highlights the directional evidence used to select features.

2. **Baseline vs model results table**
   - Reports Precision@20 and Precision@50 on the same evaluation split.

3. **Ranked Top-20 review queue**
   - Shows the highest-priority page identifiers using hashed/public-safe identifiers.
   - Includes the action score and reason code.

4. **Feature/methodology summary**
   - Documents the March feature window, April outcome window, label definition, and leakage check.

5. **Limitations summary**
   - Makes clear where the analysis should and should not be used as evidence.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.